# Lab 20 — Clustering: K-Means, Hierarchical, DBSCAN, and Evaluation

In this lab you will fit and interpret three clustering algorithms — k-means, hierarchical (agglomerative) clustering, and DBSCAN — and then use silhouette score to choose a sensible number of clusters for a real customer dataset, the same kind of workflow a data scientist runs when there are no labels to guide the decision.

**Concepts covered:** Why unsupervised evaluation has no ground truth to check against, k-means centroids and inertia (WCSS), agglomerative clustering and linkage matrices, DBSCAN's core/border/noise classification, and choosing K with the elbow method, silhouette score, and the gap statistic's null-reference intuition.

**Reference working sessions:**
- `working-sessions/unsupervised/00_unsupervised_learning_overview.ipynb`
- `working-sessions/unsupervised/01_what_is_unsupervised_learning.ipynb`
- `working-sessions/unsupervised/02_kmeans_clustering.ipynb`
- `working-sessions/unsupervised/04_hierarchical_clustering.ipynb`
- `working-sessions/unsupervised/05_dbscan.ipynb`
- `working-sessions/unsupervised/06_clustering_evaluation.ipynb`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage

from tkh_utils import (
    PALETTE, FONT, base_layout,
    check_answer, make_answer_key, make_grading_summary,
    load_mall_customers,
)

_ak = make_answer_key({
    'q1': 'B',
    'q2': 'B',
    'q3': 'B',
    'q4': 'B',
})

---
## Section A — Multiple Choice

Fill in each answer variable with the letter of the best answer (A, B, C, or D).

In [ ]:
# Q1 — Per working-sessions/unsupervised/01_what_is_unsupervised_learning.ipynb,
# why is evaluating an unsupervised clustering result fundamentally harder
# than evaluating a supervised classifier?
#
#   A) Unsupervised algorithms are always less accurate than supervised ones
#   B) There are no ground-truth labels to compare predictions against, so
#      "correct" must be judged by internal structure and domain knowledge
#      instead of a metric like accuracy
#   C) Unsupervised learning never uses distance calculations
#   D) Supervised learning doesn't require a train/test split, but
#      unsupervised learning does

q1_answer = "___"  # Replace with A, B, C, or D

assert q1_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q1_answer, _ak['q1']), \
    "Not quite — revisit working-sessions/unsupervised/" \
    "01_what_is_unsupervised_learning.ipynb and its \"What's happening?\" section."
print("✓ Question 1 correct!")

In [ ]:
# Q2 — Per working-sessions/unsupervised/02_kmeans_clustering.ipynb, k-means
# requires standardizing (scaling) features before fitting. Why?
#
#   A) sklearn's KMeans throws an error if it receives unscaled data
#   B) K-means assigns points to clusters using Euclidean distance to
#      centroids; if features are on very different scales, the
#      largest-scale feature will dominate every distance calculation
#   C) Scaling is only needed for hierarchical clustering, not k-means
#   D) Scaling changes how many clusters k-means will find, regardless of
#      the data's actual structure

q2_answer = "___"  # Replace with A, B, C, or D

assert q2_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q2_answer, _ak['q2']), \
    "Not quite — revisit working-sessions/unsupervised/02_kmeans_clustering.ipynb " \
    "and its \"Strengths and weaknesses\" table."
print("✓ Question 2 correct!")

In [ ]:
# Q3 — Per working-sessions/unsupervised/05_dbscan.ipynb, what does a label
# of -1 mean in DBSCAN's output?
#
#   A) The point belongs to cluster number -1, which sklearn always
#      creates first
#   B) The point is noise — it isn't a core point and isn't reachable from
#      any core point, so DBSCAN doesn't force it into any cluster
#   C) The algorithm failed to converge for that point
#   D) The point sits exactly on the boundary between two clusters

q3_answer = "___"  # Replace with A, B, C, or D

assert q3_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q3_answer, _ak['q3']), \
    "Not quite — revisit working-sessions/unsupervised/05_dbscan.ipynb " \
    "and its \"How it learns\" section."
print("✓ Question 3 correct!")

In [ ]:
# Q4 — Per working-sessions/unsupervised/06_clustering_evaluation.ipynb, the
# gap statistic compares your clustering's WCSS to the WCSS on a
# uniformly-random reference dataset built from the same bounding box as
# your data. What does a SMALL gap at a given K suggest?
#
#   A) The clustering is unusually good at that K
#   B) Your data looks about as clustered as pure random noise at that K —
#      there's little real structure being found
#   C) You should immediately increase K until the gap statistic reaches zero
#   D) The reference dataset was built incorrectly and should be regenerated

q4_answer = "___"  # Replace with A, B, C, or D

assert q4_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q4_answer, _ak['q4']), \
    "Not quite — revisit working-sessions/unsupervised/06_clustering_evaluation.ipynb " \
    "and its \"What's happening?\" section on the gap statistic."
print("✓ Question 4 correct!")

In [ ]:
make_grading_summary([
    (q1_answer, _ak['q1'], "Q1: Why unsupervised evaluation has no ground truth"),
    (q2_answer, _ak['q2'], "Q2: Why k-means needs scaled features"),
    (q3_answer, _ak['q3'], "Q3: What DBSCAN's -1 label means"),
    (q4_answer, _ak['q4'], "Q4: What a small gap statistic value suggests"),
], total=4)

---
## Section B — Coding Exercises

The three exercises below fit each of the three clustering algorithms from this session on synthetic data with a known structure, so you can check your results against what you already expect to find.

### B1 — K-means and inertia

Fit k-means on synthetic blobs with a known number of centers, and inspect its inertia (WCSS) and cluster sizes.

In [ ]:
# B1 — Fit k-means and inspect inertia
np.random.seed(42)
X_b1, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.65, random_state=42)
X_b1_sc = StandardScaler().fit_transform(X_b1)

kmeans_b1 = ___(n_clusters=4, n_init=10, random_state=42)   # YOUR CODE — the clustering model used throughout this notebook for the k-means algorithm
kmeans_b1.fit(___)   # YOUR CODE — the scaled features

b1_inertia = kmeans_b1.inertia_
b1_sizes = np.bincount(kmeans_b1.labels_)

print(f"Inertia (WCSS): {b1_inertia:.2f}")
print(f"Cluster sizes: {b1_sizes}")

# --- checks ---
assert len(b1_sizes) == 4, "Expected 4 clusters, matching the 4 centers used to generate the data"
assert b1_inertia < 500, "Inertia should be small once points are correctly grouped around their centroids"
print("✓ B1 complete!")

### B2 — Agglomerative clustering and the linkage matrix

Fit agglomerative (hierarchical) clustering with Ward linkage, and build the linkage matrix scipy uses to draw a dendrogram.

In [ ]:
# B2 — Fit agglomerative clustering and build the linkage matrix
np.random.seed(42)
X_b2, y_true_b2 = make_blobs(n_samples=150, centers=3, cluster_std=0.60, random_state=42)
X_b2_sc = StandardScaler().fit_transform(X_b2)

agg_b2 = AgglomerativeClustering(n_clusters=___, linkage='ward')   # YOUR CODE — the number of clusters, matching the number of true centers generated above
b2_labels = agg_b2.___(X_b2_sc)   # YOUR CODE — the method that both fits the model and returns its cluster assignments in one call

Z_b2 = linkage(___, method='ward')   # YOUR CODE — the same scaled features used to fit the clustering above

print(f"Unique clusters found: {len(set(b2_labels))}")
print(f"Linkage matrix shape: {Z_b2.shape}")

# --- checks ---
assert len(set(b2_labels)) == 3, "Expected 3 clusters, matching the 3 centers used to generate the data"
assert Z_b2.shape == (149, 4), "The linkage matrix should have one row per merge: n_samples - 1 rows, 4 columns"
print("✓ B2 complete!")

### B3 — DBSCAN with noise

Fit DBSCAN on blob data with added random noise points, and confirm it separates the real clusters from the noise.

In [ ]:
# B3 — Fit DBSCAN and count clusters vs. noise points
np.random.seed(42)
X_clean_b3, _ = make_blobs(n_samples=200, centers=3, cluster_std=0.5, random_state=42)
noise_b3 = np.random.uniform(-8, 8, size=(30, 2))
X_b3 = np.vstack([X_clean_b3, noise_b3])
X_b3_sc = StandardScaler().fit_transform(X_b3)

dbscan_b3 = DBSCAN(eps=___, min_samples=5)   # YOUR CODE — a reasonable epsilon neighborhood radius for this standardized data (try values between 0.25 and 0.4)
b3_labels = dbscan_b3.fit_predict(___)   # YOUR CODE — the scaled features

b3_n_clusters = len(set(b3_labels)) - (1 if -1 in b3_labels else 0)
b3_n_noise = np.sum(b3_labels == -1)

print(f"Clusters found: {b3_n_clusters}")
print(f"Noise points: {b3_n_noise}")

# --- checks ---
assert b3_n_clusters == 3, "Should recover the 3 real clusters — adjust eps if you get a different count"
assert b3_n_noise > 0, "The uniformly-scattered points added above should mostly be flagged as noise"
print("✓ B3 complete!")

---
## Section C — Applied Problem

A mall wants to segment its customers using Age, Annual Income, and Spending Score, but has no idea how many segments actually exist in the data. Score a range of K values with silhouette score, pick the best one, fit the final model, and visualize the result in 2D with PCA.

In [ ]:
# Section C — Full customer segmentation workflow

# --- Step 1: Load and scale ---
mall = load_mall_customers()
X_mall = mall[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].values
X_mall_sc = ___().fit_transform(___)   # YOUR CODE — a fresh instance of the feature scaler used throughout this notebook; the raw mall customer features

# --- Step 2: Try a range of K and score each with silhouette ---
k_range_c = range(2, 9)
sil_scores_c = {}
for k in k_range_c:
    km_c = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels_c = km_c.___(X_mall_sc)   # YOUR CODE — the method that both fits the model and returns its cluster assignments in one call
    sil_scores_c[k] = silhouette_score(___, labels_c)   # YOUR CODE — the scaled mall customer features

best_k = max(sil_scores_c, key=sil_scores_c.get)
print("Silhouette scores by K:", {k: round(v, 3) for k, v in sil_scores_c.items()})
print(f"Best K by silhouette: {best_k}")

# --- Step 3: Fit the final model at the chosen K ---
final_km = KMeans(n_clusters=___, n_init=10, random_state=42)   # YOUR CODE — the K selected by silhouette score above
mall['segment'] = final_km.fit_predict(X_mall_sc)

# --- Step 4: Visualize with PCA ---
pca = PCA(n_components=2, random_state=42)
X_2d = pca.___(X_mall_sc)   # YOUR CODE — the method that both fits PCA and returns the transformed 2D coordinates

fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(x=X_2d[:, 0], y=X_2d[:, 1], hue=mall['segment'], palette='deep', ax=ax)
ax.set_title(f"Mall Customer Segments (K={best_k})")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
plt.tight_layout()
plt.show()

# --- checks ---
assert best_k >= 2, "Should find at least 2 clusters"
assert mall['segment'].nunique() == best_k, "The fitted model's cluster count should match the K chosen by silhouette score"
assert X_2d.shape == (len(mall), 2), "PCA output should have one 2D row per customer"
print("✓ Section C complete!")
print(f"  Chosen K: {best_k}")

---

## Section D — Reflection

These questions are for reflection. Edit the markdown cells below each question to write your response. There are no wrong answers, we are looking for thoughtful engagement with what you have learned. Your instructor may review these.

**Question D1**

Imagine the mall instead wanted to flag customers whose behavior doesn't fit any typical segment at all (potential data errors, or truly unusual customers) rather than force every customer into one of K groups. Would you recommend switching Section C's approach from k-means to DBSCAN? Justify your answer using the "When to use it / When NOT to use it" table in `working-sessions/unsupervised/05_dbscan.ipynb`.

*Your response here...*

**Question D2**

In `working-sessions/unsupervised/06_clustering_evaluation.ipynb`, the gap statistic's decision rule is "the smallest K that produces the largest gap," not simply "whichever K happens to score highest." Suppose in a real project K=5 and K=6 produce nearly identical gap statistic values. Which would you recommend, and why? Use the notebook's overfitting-caution reasoning to justify your answer.

*Your response here...*

**Question D3**

Section C found the best K by silhouette score for the mall customer dataset. A marketing manager tells you her team can only design and execute 3-4 different campaigns this quarter. Using the "Why this matters for data science" reasoning from `working-sessions/unsupervised/06_clustering_evaluation.ipynb`, how would you reconcile the statistically "best" K with this business constraint?

*Your response here...*